# Septin intensity vs outline curvature

For each cell in a set of paired brightfield + fluorescence ND2 stills, measures outline
curvature and fluorescence at 300 points around the cell, then tests the association
within each cell (cells, not points, are the unit). Guide:
[docs/SEPTIN.md](https://github.com/widenerm/pombe-bs-detector/blob/main/docs/SEPTIN.md).

**Before you start:** *Runtime > Change runtime type > T4 GPU*, and put the ND2 folder
for one imaging day in Google Drive. Files are paired in name order, and the smaller file
of each pair is taken as brightfield. Check the pairs printed in step 3.

In [ ]:
#@title 1. Install BS-Detector (about 2 minutes)
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again"
!pip install -q "bs-detector @ git+https://github.com/widenerm/pombe-bs-detector"
!bsd --help | head -3

In [ ]:
#@title 2. Choose the data
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

INPUT = "/content/drive/MyDrive/septin/080724"  #@param {type:"string"}
#@markdown Folder of ND2 pairs for one day.
OUT = ""  #@param {type:"string"}
#@markdown Results folder. Empty = `MyDrive/bsd_septin/<folder name>`.
MASKS = ""  #@param {type:"string"}
#@markdown Optional: an existing masks file, to keep cell IDs that labels refer to. Empty = `OUT/masks.h5`.
EXCLUDE = ""  #@param {type:"string"}
#@markdown Optional: CSV of cells to leave out (e.g. dividing-cell labels with `cell_id`, `label`).
MIN_LENGTH_UM = ""  #@param {type:"string"}
#@markdown Optional: also report only cells at least this long (µm), e.g. `8.5` to leave out newly separated cells.
FLUOR_CHANNEL = 0  #@param {type:"integer"}

OUT = OUT or f'/content/drive/MyDrive/bsd_septin/{Path(INPUT).name}'
masks = f'--masks "{MASKS}"' if MASKS else ''
filters = (f'--exclude "{EXCLUDE}" ' if EXCLUDE else '') + (f'--min-length-um {MIN_LENGTH_UM}' if MIN_LENGTH_UM else '')
print('input:', INPUT); print('results:', OUT)

In [ ]:
#@title 3. Segment (once; fields already in the masks file are kept as they are)
!bsd septin segment "{INPUT}" --out "{OUT}" {masks}

In [ ]:
#@title 4. Measure curvature and fluorescence along each outline
!bsd septin measure "{INPUT}" --out "{OUT}" {masks} --fluor-channel {FLUOR_CHANNEL}

In [ ]:
#@title 5. Statistics
!bsd septin stats "{OUT}" {filters}

In [ ]:
#@title 6. Report
from IPython.display import Image, Markdown, display
out = Path(OUT)
display(Markdown((out / 'statistics' / 'statistics_report.md').read_text()))
if (out / 'statistics_comparison.md').exists():
    display(Markdown((out / 'statistics_comparison.md').read_text()))
display(Image(str(out / 'statistics' / 'smoothing_sensitivity.png'), width=600))
for png in sorted((out / 'frame_overlays').glob('*.png'))[:1]:
    display(Image(str(png), width=600))
print('Everything is saved in', OUT)